# 2. One act: its text, its files and its checksums

Each act has raw files in the bucket
[Sarverott/raw-law-of-poland](https://huggingface.co/buckets/Sarverott/raw-law-of-poland)
(details, PDF, HTML) and markdown made from its HTML in the dataset
(`texts/`). The dataset also records the checksum of every raw file at the
moment it was gathered (`files/`), so a copy can be compared with what the
source gave.

The dataset is public: nothing here needs a token. Run it with
`uv run --project examples --with jupyterlab jupyter lab examples/notebooks` (or the marimo version: `task examples:convert`, then
version: `task examples:convert`, then
`uv run --project examples marimo edit examples/marimo`).

In [ ]:
import hashlib
import json

import pandas as pd
import requests
from huggingface_hub import hf_hub_download

DATASET = "Sarverott/polish_national_law_docs"
BUCKET = "https://huggingface.co/buckets/Sarverott/raw-law-of-poland/resolve"
HEADERS = {"User-Agent": "P-jol-roll examples (+https://github.com/Sarverott/P-jol-roll)"}

JOURNAL, YEAR, POS = "DU", 2016, 1
acts = pd.read_json(
    hf_hub_download(DATASET, f"acts/{JOURNAL}/{YEAR}.jsonl", repo_type="dataset"),
    lines=True,
)
act = acts[acts["pos"] == POS].iloc[0]
print(act["displayAddress"], "-", act["title"])

## Its details, from the bucket

The file is named by the act's address (`WDU20160000001`).

In [ ]:
path = f"{JOURNAL}/{YEAR}/{act['address']}"
details = requests.get(f"{BUCKET}/{path}.json", headers=HEADERS, timeout=60).json()
{key: details.get(key) for key in ("ELI", "status", "inForce", "entryIntoForce", "keywords")}

## What it changes and what carries it out

In [ ]:
pd.DataFrame(
    [
        {"tie": kind, "act": other["id"]}
        for kind, others in (details.get("references") or {}).items()
        for other in others
    ]
)

## Its text as markdown

In [ ]:
try:
    markdown_file = hf_hub_download(DATASET, f"texts/{path}.md", repo_type="dataset")
    with open(markdown_file, encoding="utf-8") as file:
        print(file.read()[:1500])
except Exception as error:  # not every act has HTML to make markdown from
    print("no markdown for this act:", error)

## The PDF, checked against the record made when it was gathered

In [ ]:
try:
    records = pd.read_json(
        hf_hub_download(DATASET, f"files/{JOURNAL}/{YEAR}.jsonl", repo_type="dataset"),
        lines=True,
    )
    record = records[records["path"] == f"{path}.pdf"].iloc[-1]
    pdf = requests.get(f"{BUCKET}/{path}.pdf", headers=HEADERS, timeout=120).content
    print("recorded:", record["sha256"])
    print("now:     ", hashlib.sha256(pdf).hexdigest())
    print("the same" if hashlib.sha256(pdf).hexdigest() == record["sha256"] else "DIFFERENT")
except Exception as error:  # the records of this year may not be on Hugging Face yet
    print("no record to compare with:", error)